# S0 — 02.03: Gradient Descent for Regression

**Module:** 02 — Supervised Regression  
**Prerequisites:** 01.01–01.06, 02.01, and 02.02  
**Difficulty:** introductory  
**Estimated time:** 45–60 minutes

> **Learner question:** How can repeated parameter updates reduce regression MSE, and how does that iterative result relate to closed-form OLS?

This is a deterministic, synthetic, dimensionless demonstration. Its short trace describes this small table only; it is not evidence of real-world convergence or model performance.

## S1 — Prediction rule and parameter vector

With two features, the model is `y_hat = b + X @ beta`. Add a column of ones so `A @ theta` includes the intercept, where `theta = [b, w1, w2]`.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

RTOL = 0.0
ATOL = 1e-12
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [1.0, 2.0]], dtype=np.float64)
y = np.array([5.0, 1.0, 4.0, 8.0, 3.0], dtype=np.float64)
A = np.column_stack((np.ones(X.shape[0], dtype=np.float64), X))
theta = np.zeros(3, dtype=np.float64)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("A shape:", A.shape)
print("theta shape:", theta.shape)

## S2 — Residuals, RSS, and MSE

For residuals `r = y_hat - y`, `RSS = sum(r**2)` and `MSE = RSS / n_samples`. The objective is MSE; RSS remains visible for continuity with 02.02.

In [ ]:
predictions = np.asarray(A @ theta, dtype=np.float64)
residuals = np.asarray(predictions - y, dtype=np.float64)
rss = np.float64(np.sum(residuals**2, dtype=np.float64))
mse = np.float64(rss / y.size)
print("predictions:", predictions)
print("residuals:", residuals)
print("RSS:", rss)
print("MSE:", mse)
np.testing.assert_allclose(mse, 23.0, rtol=RTOL, atol=ATOL)

## S3–S4 — The gradient

For MSE, the intercept derivative is `(2/n) * sum(r)` and each coefficient derivative is `(2/n) * sum(r*x_j)`. In matrix form:

`gradient(theta) = (2/n) * A.T @ (A @ theta - y)`

The gradient points uphill, so descent subtracts it.

In [ ]:
gradient = np.asarray((np.float64(2.0) / y.size) * (A.T @ residuals), dtype=np.float64)
print("gradient at theta_0:", gradient)
np.testing.assert_allclose(gradient, [-8.4, -11.2, -7.6], rtol=RTOL, atol=ATOL)

## S5–S7 — Learning rate and one update

Use the frozen learning rate `eta = 0.1` and update all parameters simultaneously:

`theta_next = theta - eta * gradient(theta)`

In [ ]:
eta = np.float64(0.1)
theta_1 = np.asarray(theta - eta * gradient, dtype=np.float64)
print("theta_1:", theta_1)
np.testing.assert_allclose(theta_1, [0.84, 1.12, 0.76], rtol=RTOL, atol=ATOL)

## S8 — Five deterministic updates

The step count is frozen for reproducibility, not presented as a general stopping rule. A falling loss here is evidence about this data, initialization, learning rate, and five updates only.

In [ ]:
theta = np.zeros(3, dtype=np.float64)
learning_rate = np.float64(0.1)
theta_trace = [theta.copy()]
gradient_trace = []
mse_trace = []
for _ in range(5):
    predictions = np.asarray(A @ theta, dtype=np.float64)
    residuals = np.asarray(predictions - y, dtype=np.float64)
    mse = np.float64(np.mean(residuals**2, dtype=np.float64))
    gradient = np.asarray((np.float64(2.0) / y.size) * (A.T @ residuals), dtype=np.float64)
    mse_trace.append(mse)
    gradient_trace.append(gradient)
    theta = np.asarray(theta - learning_rate * gradient, dtype=np.float64)
    theta_trace.append(theta.copy())
predictions = np.asarray(A @ theta, dtype=np.float64)
residuals = np.asarray(predictions - y, dtype=np.float64)
mse_trace.append(np.float64(np.mean(residuals**2, dtype=np.float64)))
print("theta trace:\n", np.asarray(theta_trace))
print("MSE trace:", np.asarray(mse_trace))
np.testing.assert_allclose(mse_trace, [23.0, 5.7552, 3.25568256, 2.574780076032, 2.168511051300863, 1.8446924257768553], rtol=RTOL, atol=ATOL)

## S9 — Shape-aware NumPy implementation

The public verifier and this notebook use NumPy arrays with `np.float64`, explicit dimensions, finite values, matching sample counts, a parameter vector of shape `(n_features + 1,)`, a positive finite learning rate, and a non-negative integer iteration count. Invalid inputs should raise explicit errors.

## S10 — Independent cross-check

The companion `verify_hand_calc.py` independently recomputes gradients, updates, predictions, residuals, and losses. It does not import this notebook or a primary implementation. Assertions use `rtol=0.0` and `atol=1e-12`.

## S11 — Compare with OLS

OLS solves the same least-squares objective directly for this example. Gradient descent reaches a nearby point after five prescribed updates, not the exact minimizer.

In [ ]:
x_bar = np.mean(X, axis=0, dtype=np.float64)
y_bar = np.float64(np.mean(y, dtype=np.float64))
centered_X = X - x_bar
centered_y = y - y_bar
beta = np.linalg.solve(centered_X.T @ centered_X, centered_X.T @ centered_y)
intercept = np.float64(y_bar - x_bar @ beta)
ols_theta = np.concatenate((np.array([intercept], dtype=np.float64), beta))
ols_predictions = np.asarray(A @ ols_theta, dtype=np.float64)
ols_residuals = np.asarray(ols_predictions - y, dtype=np.float64)
print("OLS theta:", ols_theta)
print("OLS RSS:", np.sum(ols_residuals**2, dtype=np.float64))
print("OLS MSE:", np.mean(ols_residuals**2, dtype=np.float64))
np.testing.assert_allclose(ols_theta, [1.7, 3.5, -1.0], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(np.mean(ols_residuals**2), 0.06, rtol=RTOL, atol=ATOL)
model = LinearRegression().fit(X, y)
np.testing.assert_allclose(model.intercept_, intercept, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(model.coef_, beta, rtol=RTOL, atol=ATOL)
print("LinearRegression reference agrees")

![Training MSE across five gradient-descent updates](assets/gradient_descent_loss.png)

**Text equivalent:** MSE is 23.0, 5.7552, 3.25568256, 2.574780076032, 2.168511051300863, and 1.8446924257768553 for states 0–5. The chart is supplementary and does not rely on color or interaction.

## S12 — Recap and handoff

Gradient descent repeatedly subtracts a scaled MSE gradient; closed-form OLS solves the same least-squares problem directly. This small convex trace does not establish general convergence or generalisation. In 02.04, polynomial feature construction changes the design matrix while this distinction between model and optimizer remains important.